In [1]:
!pip3 install pandas openpyxl
import pandas as pd
import numpy as np

df = pd.read_excel('dataset.xlsx', engine='openpyxl')
df = df.iloc[:, :-1]
df.head(5)


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\maxim\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


,ФИО,Кол-во Детей,Unnamed: 2,Вареники,Блины,Манты,Гедза,Мясо,Зелень,Фаст-фуд,...,Топливо,авто,Яндекс,Сбер,Вуш,Кап.ремонт,Мусор,Вода,Свет,Отопление
0,Абрамов Фёдор Степанович,5,4771,6441,2129,7262,6571,6571,4226,4985,...,5090,897,7938,5679,3479,7336,108,1409,9445,3812
1,Абрамов Андрей Даниилович,5,2259,9288,3941,6429,9012,5653,3221,3619,...,3314,1186,786,7522,3326,5589,9161,6603,8131,4567
2,Абрамов Михаил Игоревич,1,4623,4318,2799,5582,8064,1091,5836,6773,...,1493,8121,5875,138,9458,4004,5965,3088,5568,9117
3,Абрамов Глеб Львович,2,7853,642,6408,8347,4399,3337,2268,4969,...,3199,3341,385,3867,6963,9375,4460,4017,5387,3024
4,Абрамов Николай Артёмович,2,4903,6350,336,1969,8981,6215,2070,7829,...,6022,1769,8516,4184,4447,7437,6729,8728,8604,9872


# Задание 1 Как выглядит «типичная семья»?

In [2]:
categories = ["Свет", "Мусор", "Мясо", "Отопление", "авто"]
categories = [c for c in categories if c in df.columns]
categories

rows = []
for c in categories:
    s = df[c].dropna().astype(float)
    m = s.mode()                       # may return several values

    rows.append({
        "Категория":  c,
        "Среднее":    s.mean(),                       # x̄ = (1/n)·Σxᵢ
        "Медиана":    s.median(),
        "Мода":       m.iloc[0] if len(m) else np.nan,
        "Дисперсия":  s.var(ddof=1),                  # s² (sample)
        "σ":          s.std(ddof=1),                  # s  (sample)
        "Min":        s.min(),
        "Max":        s.max(),
        "Асимметрия": s.skew(),                       # helps answer Q2
    })

res = pd.DataFrame(rows).set_index("Категория")
res.head(10)

,Среднее,Медиана,Мода,Дисперсия,σ,Min,Max,Асимметрия
Категория,,,,,,,,
Мусор,5050.6056,5048.0,973.0,8.303890e+06,2881.647173,100.0,9998.0,-0.003473
Мясо,5032.6271,5048.0,468.0,8.131123e+06,2851.512348,100.0,9999.0,-0.005683
Отопление,5027.7399,5020.0,291.0,8.181543e+06,2860.339659,101.0,10000.0,0.005184
авто,5008.0364,4974.5,2849.0,8.136246e+06,2852.410581,101.0,9998.0,0.021177


In [3]:
# distance between mean and median -> answers Question 1
res["|Среднее − Медиана|"] = (res["Среднее"] - res["Медиана"]).abs()

print("=" * 90)
print("ТАБЛИЦА 1. Описательные статистики по категориям расходов")
print("=" * 90)
print(res.round(2).to_string())

ТАБЛИЦА 1. Описательные статистики по категориям расходов
           Среднее  Медиана    Мода   Дисперсия        σ    Min      Max  Асимметрия  |Среднее − Медиана|
Категория                                                                                                
Мусор      5050.61   5048.0   973.0  8303890.43  2881.65  100.0   9998.0       -0.00                 2.61
Мясо       5032.63   5048.0   468.0  8131122.67  2851.51  100.0   9999.0       -0.01                15.37
Отопление  5027.74   5020.0   291.0  8181542.96  2860.34  101.0  10000.0        0.01                 7.74
авто       5008.04   4974.5  2849.0  8136246.12  2852.41  101.0   9998.0        0.02                33.54


# Задание 2. Какие расходы связаны друг с другом?

In [4]:
exclude_cols = ["Кол-во Детей", "ФИО"]

numeric_cols = [
    c for c in df.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])
]
cat_cols = numeric_cols[1:25]

print("Выбранные категории:")
print(cat_cols)

data = df[cat_cols].copy()
data.head(10)

Выбранные категории:
['Вареники', 'Блины', 'Манты', 'Гедза', 'Мясо', 'Зелень', ' Фаст-фуд', 'Роллы', ' Пицца', 'Черноголовка', ' Пепси', 'Колла', 'Фанта', 'Спрайт', 'Ибупрофен', 'Ношпа', 'Омепразол', 'Фуросимед', 'Амплодин', 'Хоккей', 'Активный отдых', 'Футбол', 'Бассейн  ', 'Танцы']


,Вареники,Блины,Манты,Гедза,Мясо,Зелень,Фаст-фуд,Роллы,Пицца,Черноголовка,...,Ибупрофен,Ношпа,Омепразол,Фуросимед,Амплодин,Хоккей,Активный отдых,Футбол,Бассейн,Танцы
0,6441,2129,7262,6571,6571,4226,4985,4384,8029,7952,...,4548,3368,8804,9391,2700,9035,4041,9361,8685,1314
1,9288,3941,6429,9012,5653,3221,3619,5858,7614,629,...,7875,146,1754,6118,2314,327,2139,9393,193,5748
2,4318,2799,5582,8064,1091,5836,6773,3647,6538,7190,...,6249,689,9601,8078,6471,2606,8451,6959,5347,9835
3,642,6408,8347,4399,3337,2268,4969,4511,1421,526,...,8332,9905,1487,2251,5608,2340,2198,5912,7761,420
4,6350,336,1969,8981,6215,2070,7829,5794,2222,2255,...,1493,3622,2118,3047,6369,8563,4650,8981,9485,1380
5,865,7443,4357,1051,3771,9337,6477,8128,2179,4818,...,7944,5663,883,8604,6215,3741,6313,2016,1182,1956
6,5074,7901,746,109,190,172,7907,7408,8136,868,...,3447,1239,9234,7834,3015,6158,2068,5072,6312,101
7,3241,3226,7283,7287,4442,8154,5897,1387,2787,6734,...,5450,5669,8589,1047,1736,8460,4338,6270,9722,9329
8,5735,3688,4375,5991,7113,5127,6880,6541,3297,9880,...,1597,2153,1764,8051,6925,4212,6745,7191,6046,1702
9,5430,7895,4571,8510,5567,5110,3016,9204,3117,8729,...,5635,6137,4480,9218,6884,2227,6398,3987,4826,3980


In [5]:
from scipy import stats

corr_matrix = data.corr(method="pearson")

rows = []

for i in range(len(cat_cols)):
    for j in range(i + 1, len(cat_cols)):
        c1, c2 = cat_cols[i], cat_cols[j]
        
        pair = data[[c1, c2]].dropna()
        
        # Нужно хотя бы 3 наблюдения и вариация в обеих переменных
        if len(pair) >= 3 and pair[c1].nunique() > 1 and pair[c2].nunique() > 1:
            r, p = stats.pearsonr(pair[c1], pair[c2])
            rows.append({
                "var1": c1,
                "var2": c2,
                "r": r,
                "p_value": p,
                "n": len(pair)
            })

pairs = pd.DataFrame(rows)

# Поправка Бонферрони на множественные сравнения
m = len(pairs)
pairs["p_bonf"] = np.minimum(pairs["p_value"] * m, 1)

pairs = pairs.sort_values("r", ascending=False)

# =========================
# 5. ТОП-3 ПОЛОЖИТЕЛЬНЫЕ И ТОП-3 ОТРИЦАТЕЛЬНЫЕ
# =========================
top_pos = pairs.head(3)
top_neg = pairs.sort_values("r", ascending=True).head(3)

print("\n=== 3 самые сильные ПОЛОЖИТЕЛЬНЫЕ корреляции ===")
print(top_pos[["var1", "var2", "r", "p_value"]].to_string(index=False))

print("\n=== 3 самые сильные ОТРИЦАТЕЛЬНЫЕ корреляции ===")
print(top_neg[["var1", "var2", "r", "p_value"]].to_string(index=False))

# =========================
# 6. ВСЕ ПАРЫ ДЛЯ ПОИСКА НЕОЖИДАННОЙ КОРРЕЛЯЦИИ
# =========================
print("\nВсе пары, отсортированные по |r|:")
pairs["abs_r"] = pairs["r"].abs()
print(
    pairs.sort_values("abs_r", ascending=False)
    [["var1", "var2", "r", "p_value"]]
    .head(20)
    .to_string(index=False)
)



=== 3 самые сильные ПОЛОЖИТЕЛЬНЫЕ корреляции ===
  var1      var2        r  p_value
 Пепси Ибупрофен 0.035558 0.000376
 Фанта    Спрайт 0.025064 0.012194
 Пепси     Колла 0.024293 0.015126

=== 3 самые сильные ОТРИЦАТЕЛЬНЫЕ корреляции ===
  var1   var2         r  p_value
 Пепси Спрайт -0.038368 0.000124
 Блины Хоккей -0.020805 0.037487
 Колла Хоккей -0.020792 0.037601

Все пары, отсортированные по |r|:
     var1      var2         r  p_value
    Пепси    Спрайт -0.038368 0.000124
    Пепси Ибупрофен  0.035558 0.000376
    Фанта    Спрайт  0.025064 0.012194
    Пепси     Колла  0.024293 0.015126
    Пепси Омепразол  0.024067 0.016094
 Вареники Фуросимед  0.022234 0.026187
Бассейн       Танцы  0.021851 0.028883
    Гедза    Спрайт  0.021290 0.033257
    Блины    Хоккей -0.020805 0.037487
    Колла    Хоккей -0.020792 0.037601
    Манты Фуросимед  0.020703 0.038430
 Вареники     Роллы  0.019132 0.055734
    Блины    Зелень  0.019055 0.056721
     Мясо Омепразол  0.018897 0.058814
    Колл

# 

# Задание 3. Можно ли предсказать расходы семьи?

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_excel("dataset.xlsx") 

exclude_cols = ["Кол-во Детей", "ФИО"]

numeric_cols = [c for c in df.columns if c not in exclude_cols]

cat_cols = numeric_cols[1:]
df = df[cat_cols].copy()


target = "Вода"  

corrs = df.corr()[target].drop(target).sort_values(key=np.abs, ascending=False)
best_feature = corrs.index[0]
print("\nMost correlated feature:", best_feature)

X_simple = df[[best_feature]]
y = df[target]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y, test_size=0.2, random_state=42)

simple = LinearRegression()
simple.fit(X_train_s, y_train_s)
y_pred_s = simple.predict(X_test_s)

print("\n--- SIMPLE LINEAR REGRESSION ---")
print("Coefficient:", simple.coef_[0])
print("R2 (test):", r2_score(y_test_s, y_pred_s))
print("MAE (test):", mean_absolute_error(y_test_s, y_pred_s))
print("RMSE (test):", np.sqrt(mean_squared_error(y_test_s, y_pred_s)))


# 7. Multiple regression: top 5 correlated features
top5 = corrs.index[:5].tolist()

print("\nMost correlated features: ", top5)

X_multi = df[top5]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

multi = LinearRegression()
multi.fit(X_train_m, y_train_m)
y_pred_m = multi.predict(X_test_m)

print("\n--- MULTIPLE LINEAR REGRESSION ---")
print("Coefficients:")
for feature, coef in zip(top5, multi.coef_):
    print(f"{feature}: {coef}")

print("R2 (test):", r2_score(y_test_m, y_pred_m))
print("MAE (test):", mean_absolute_error(y_test_m, y_pred_m))
print("RMSE (test):", np.sqrt(mean_squared_error(y_test_m, y_pred_m)))

# 8. Comparison table
comparison = pd.DataFrame({
    "Model": ["Simple (1 feature)", "Multiple (5 features)"],
    "R2_test": [
        r2_score(y_test_s, y_pred_s),
        r2_score(y_test_m, y_pred_m)
    ],
    "MAE_test": [
        mean_absolute_error(y_test_s, y_pred_s),
        mean_absolute_error(y_test_m, y_pred_m)
    ],
    "RMSE_test": [
        np.sqrt(mean_squared_error(y_test_s, y_pred_s)),
        np.sqrt(mean_squared_error(y_test_m, y_pred_m))
    ]
})

print("\n--- COMPARISON ---")
print(comparison)


Most correlated feature: Фанта

--- SIMPLE LINEAR REGRESSION ---
Coefficient: -0.021057766968670052
R2 (test): 0.0013343877862548315
MAE (test): 2476.141910467077
RMSE (test): 2862.931631949408

Most correlated features:  ['Фанта', 'Топливо', 'Зелень', ' Пицца', 'Мусор']

--- MULTIPLE LINEAR REGRESSION ---
Coefficients:
Фанта: -0.02072238170398333
Топливо: 0.026971453019781075
Зелень: 0.025681222325376844
 Пицца: -0.024185856630819326
Мусор: -0.018797525816760727
R2 (test): -0.0013086529938273195
MAE (test): 2477.99947296401
RMSE (test): 2866.7176064662103

--- COMPARISON ---
                   Model   R2_test     MAE_test    RMSE_test
0     Simple (1 feature)  0.001334  2476.141910  2862.931632
1  Multiple (5 features) -0.001309  2477.999473  2866.717606


# Задание 4. Насколько можно доверять выборке?

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# ==============================
# 1. Загрузка набора данных
# ==============================
df = pd.read_excel("dataset.xlsx")   # или: pd.read_csv("your_file.csv")

column_list = df.columns.tolist()
column_list = list(df.columns)
column_list = column_list[2:]

df['общие_расходы'] = df[column_list].sum(axis=1)


population = df["общие_расходы"].to_numpy()
pop_mean = population.mean()
N = len(population)

print(f"Объём генеральной совокупности: {N}")
print(f"Истинное среднее генеральной совокупности: {pop_mean:,.2f}")

# ==============================
# 3. Выборки объёмом 50, 100, 500, 1000
# ==============================
rng = np.random.default_rng(42) 

rows = []

for n in [50, 100, 500, 1000]:
    sample = rng.choice(population, size=n, replace=False)
    sample_mean = sample.mean()
    error = sample_mean - pop_mean
    error_pct = abs(error) / pop_mean * 100

    rows.append({
        "Объём выборки": n,
        "Среднее": sample_mean,
        "Ошибка": error,
        "Ошибка, %": error_pct
    })

table = pd.DataFrame(rows)
print("\nРезультаты по выборкам:")
print(table.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

# ==============================
# 4. Повторение выборки объёмом 100 десять раз
# ==============================
means_100 = []
samples_100 = []

for i in range(10):
    sample = rng.choice(population, size=100, replace=False)
    samples_100.append(sample)
    means_100.append(sample.mean())

q2 = pd.DataFrame({
    "Повтор": range(1, 11),
    "Среднее по 100": means_100
})

print("\nДесять повторных выборок объёмом 100:")
print(q2.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

Объём генеральной совокупности: 10000
Истинное среднее генеральной совокупности: 186,632.98

Результаты по выборкам:
 Объём выборки    Среднее    Ошибка  Ошибка, %
            50 183,907.42 -2,725.56       1.46
           100 189,786.46  3,153.48       1.69
           500 187,908.31  1,275.33       0.68
          1000 186,832.53    199.55       0.11

Десять повторных выборок объёмом 100:
 Повтор  Среднее по 100
      1      186,046.34
      2      184,322.74
      3      184,975.74
      4      188,022.28
      5      186,035.35
      6      187,034.96
      7      187,855.71
      8      183,337.27
      9      189,102.97
     10      184,554.96


# Задание 5. Найдите «необычные семьи»